# AMD Compute Workflow — Patient-Similarity Embeddings & Offline Reasoning QA

Runs on the AMD Developer Cloud Jupyter environment. This notebook is a complementary,
offline part of the Diabetic Complication Swarm project — it does NOT replace the live
Featherless-powered backend agent. See `amd_compute/README.md` for the full explanation.

**Run this notebook top to bottom on the AMD instance. Do not clear cell outputs before
committing** — the device-info output in the next cell is part of the Track 3 proof.

## 1. Confirm we're running on AMD hardware

In [ ]:
import subprocess, sys, time, json

print('--- rocm-smi ---')
try:
    print(subprocess.run(['rocm-smi'], capture_output=True, text=True, timeout=10).stdout)
except Exception as e:
    print(f'rocm-smi not available in this shell: {e}')

import torch
print('torch version:', torch.__version__)
print('cuda/rocm available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device name:', torch.cuda.get_device_name(0))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

## 2. Load the NHANES-derived patient dataset

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path('../backend/real_patients.csv')
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

patients = pd.read_csv(DATA_PATH)
print(f'Loaded {len(patients)} patients')
patients.head()

## 3. Encode each patient's lab profile into an embedding (runs on the AMD GPU)

We turn each patient's labs into a short natural-language profile and embed it with a
sentence-transformer model. This gives a vector we can use for nearest-neighbor
'similar patient' lookups in the dashboard — a feature the live Featherless agent
does not compute itself.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)

def profile_text(row) -> str:
    return (
        f"Age {row['age']}, {row['sex']}, {row['years_with_diabetes']} years with diabetes. "
        f"A1c {row['a1c_percent']}%, eGFR {row['egfr']}, UACR {row['uacr_mg_g']} mg/g, "
        f"creatinine {row['creatinine_mg_dl']} mg/dL, LDL {row['ldl_mg_dl']}, HDL {row['hdl_mg_dl']}, "
        f"triglycerides {row['triglycerides_mg_dl']}, systolic BP {row['systolic_bp']}."
    )

texts = [profile_text(r) for _, r in patients.iterrows()]

start = time.perf_counter()
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True)
elapsed = time.perf_counter() - start
print(f'Encoded {len(texts)} patients in {elapsed:.2f}s on {DEVICE}')

np.save(OUT_DIR / 'patient_embeddings.npy', embeddings)
with open(OUT_DIR / 'patient_index.json', 'w') as f:
    json.dump({'patient_ids': patients['patient_id'].astype(str).tolist()}, f, indent=2)

print('Saved embeddings to outputs/patient_embeddings.npy')

## 4. Sanity check: nearest neighbors for a sample patient

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sample_idx = 0
sims = cosine_similarity(embeddings[[sample_idx]], embeddings)[0]
top5 = sims.argsort()[::-1][1:6]

print(f"Patients most similar to {patients.iloc[sample_idx]['patient_id']}:")
for idx in top5:
    print(f"  {patients.iloc[idx]['patient_id']}  (similarity {sims[idx]:.3f})")

## 5. Load live Featherless-backend reasoning + judge it with a LOCAL model on this AMD GPU

Loads `reasoning_batch_export.json`, produced by running
`backend/export_reasoning_batch.py` against the live Featherless-backed pipeline
(same `run_pipeline.py` / `specialists.py` / `agent_core.py` your deployed backend
uses). Upload that JSON file into this Jupyter environment next to this notebook
before running this cell (or re-run the export script here if this instance can
reach Featherless directly).

The QA judge itself is a model pulled and run **locally via Ollama on THIS AMD
instance's GPU** (not Featherless, not any hosted API) - this is the genuine AMD
compute step: real inference work, done locally, scoring real output from the
live product.

**`JUDGE_MODEL` is switchable** between any locally-pulled Ollama model — set it to
`'llama3'` or `'gemma2'` (Gemma, run locally) in the next cell. Each run writes its
own `outputs/reasoning_eval_report_<model>.json` (not overwritten by the other), so
you can run this cell once per model and compare both — the dashboard's AMD Compute
panel lets you toggle between whichever ones you've actually run.

In [ ]:
import subprocess, time as _time

# Any locally-pullable Ollama model. Set to 'llama3' or 'gemma2' (Gemma, run
# locally on this AMD instance's GPU — no external API involved either way).
JUDGE_MODEL = 'llama3'

subprocess.Popen(['ollama', 'serve'])
_time.sleep(5)
pull = subprocess.run(['ollama', 'pull', JUDGE_MODEL], capture_output=True, text=True)
print(pull.stdout[-500:] if pull.stdout else '(pull already present)')

In [ ]:
EXPORT_PATH = Path('reasoning_batch_export.json')
if not EXPORT_PATH.exists():
    EXPORT_PATH = Path('../amd_compute/reasoning_batch_export.json')

with open(EXPORT_PATH) as f:
    export_data = json.load(f)

reasoning_records = export_data['records']
print(f"Loaded {len(reasoning_records)} live specialist reasoning records "
      f"(source provider: {export_data['provider']}, {export_data['n_patients']} patients)")

In [ ]:
JUDGE_PROMPT_TEMPLATE = (
    "You are auditing a clinical risk-scoring explanation. Judge ONLY the reasoning "
    "below, not the underlying medicine.\n\n"
    "Specialist: {specialist}\nRisk score given: {risk_score}\nReasoning: {reasoning}\n\n"
    "Answer in exactly this format, three lines, nothing else:\n"
    "CITES_VALUE: yes/no\n"
    "CONSISTENT_WITH_SCORE: yes/no\n"
    "ONE_LINE_CRITIQUE: <your critique>"
)

def judge_with_local_model(record: dict) -> dict:
    """Runs the LOCAL Ollama model (on this AMD instance's GPU) as a judge over
    one specialist's reasoning text exported from the live Featherless backend.
    This is the actual AMD-hardware inference call - everything upstream of this
    (the reasoning text itself) already happened on Featherless before export.
    """
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        specialist=record['specialist'],
        risk_score=record['risk_score'],
        reasoning=record['reasoning'],
    )
    run = subprocess.run(['ollama', 'run', JUDGE_MODEL, prompt], capture_output=True, text=True, timeout=60)
    verdict_text = run.stdout.strip()
    return {
        **record,
        'judge_model': JUDGE_MODEL,
        'judge_verdict_raw': verdict_text,
        'passed_qa': 'CITES_VALUE: yes' in verdict_text and 'CONSISTENT_WITH_SCORE: yes' in verdict_text,
    }

start = _time.perf_counter()
qa_results = [judge_with_local_model(r) for r in reasoning_records]
judge_elapsed = _time.perf_counter() - start
print(f'Judged {len(qa_results)} records with local model {JUDGE_MODEL} in {judge_elapsed:.1f}s on {DEVICE}')

report = {
    'judge_model': JUDGE_MODEL,
    'source_provider': export_data['provider'],
    'n_samples': len(qa_results),
    'n_passed': sum(r['passed_qa'] for r in qa_results),
    'judge_elapsed_seconds': judge_elapsed,
    'results': qa_results,
}

# Slug the model name into the filename (e.g. 'llama3' -> reasoning_eval_report_llama3.json,
# 'gemma2' -> reasoning_eval_report_gemma2.json) so running this cell once per model doesn't
# overwrite the other's results — the dashboard panel can then toggle between whichever
# model-specific reports actually exist in outputs/.
model_slug = JUDGE_MODEL.replace(':', '_').replace('/', '_')
report_path = OUT_DIR / f'reasoning_eval_report_{model_slug}.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Wrote {report_path}')
print(json.dumps({k: v for k, v in report.items() if k != 'results'}, indent=2))

## 6. Write run log for judges

In [ ]:
with open(OUT_DIR / 'run_log.txt', 'w') as f:
    f.write(f'Device used: {DEVICE}\n')
    f.write(f'Patients embedded: {len(patients)}\n')
    f.write(f'Embedding time: {elapsed:.2f}s\n')
    f.write(f'QA samples scored: {len(qa_results)}\n')
print('Wrote outputs/run_log.txt')